<a href="https://colab.research.google.com/github/anilpomar/Full-Stack-Gen-AI-BootCamp-KrishNaik-/blob/main/Assignment_Stage1_Stage2_finetuning1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Let torch stay at Colab's native version - do NOT pin it

!pip uninstall -y transformers tokenizers trl peft accelerate bitsandbytes unsloth unsloth_zoo huggingface_hub

# !pip install "transformers==4.55.4" "tokenizers>=0.21,<0.22" "huggingface_hub>=0.34.0,<1.0" --no-deps
# !pip install "trl==0.19.0" "peft" "accelerate" "bitsandbytes" --no-deps
# !pip install "unsloth" "unsloth_zoo" --no-deps
!pip install "transformers==4.57.6" "trl==0.24.0" "peft>=0.18.0" "accelerate>=0.34.1" "datasets>=3.4.1,<4.4.0" "huggingface_hub>=0.34.0" "sentencepiece>=0.2.0" "bitsandbytes" "unsloth" "unsloth_zoo>=2026.2.1"
!pip install PyMuPDF


Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
Found existing installation: huggingface_hub 1.20.1
Uninstalling huggingface_hub-1.20.1:
  Successfully uninstalled huggingface_hub-1.20.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 154.6 MB/s e

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 107.0 MB/s eta 0:00:00


In [1]:

import transformers, torch
print(transformers.__version__, torch.__version__)   # MUST read 4.55.4 and torch 2.10.x


4.57.6 2.10.0+cu128


In [2]:
!pip show unsloth

Name: unsloth
Version: 2026.7.2
Summary: 2-5X faster training, reinforcement learning & finetuning
Home-page: https://unsloth.ai
Author: Unsloth AI team
Author-email: info@unsloth.ai
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, bitsandbytes, datasets, diffusers, hf_transfer, huggingface_hub, nest-asyncio, numpy, packaging, peft, protobuf, psutil, pydantic, pyyaml, sentencepiece, torch, torchvision, tqdm, transformers, triton, trl, typer, tyro, unsloth_zoo, wheel, xformers
Required-by: 


In [3]:
import unsloth   # unsloth FIRST, before trl/transformers, per its own warning
print("unsloth OK")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1427: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth OK


In [4]:

import  trl

import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset, load_dataset


from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

DPO patch applied.
GPU: NVIDIA L4


In [41]:
from dataclasses import dataclass
# -------------------------
# 3. File paths
# -------------------------

non_instruction_data_path = "/content/non_instruction_dataset.pdf"
instruction_data_path = "/content/instruction_dataset.jsonl"
preference_data_path = "/content/preference_dataset.jsonl"

for path in [non_instruction_data_path, instruction_data_path, preference_data_path]:
  if not os.path.exists(path):
      raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")

# -------------------------
# 4. Configuration
# -------------------------
BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

MAX_SEQ_LENGTH =2048 #512
SEED =3407 # was 42
MIN_CHARS_PER_PARAGRAPH = 80

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0

BATCH_SIZE = 2    #This is per_device_train_batch_size
GRAD_ACCUM_STEPS =8  #was 2 #16  This is gradient_accumulation_steps
WARMUP_STEPS = 5 #10
LOGGING_STEPS = 1

# Classroom demo steps. Increase for serious training.
STAGE1_MAX_STEPS =100 # was  15
STAGE2_MAX_STEPS = 150
STAGE3_MAX_STEPS = 20

STAGE1_LR = 5e-5       #This is learning_rate
STAGE2_LR = 2e-4        #This is learning_rate
STAGE3_LR = 5e-5       # This is learning_rate

DPO_BETA = 0.1

OUTPUT_ROOT = "/content/unsloth_pharma_merge_reload_outputs"

non_instruction_ADAPTER_DIR = f"{OUTPUT_ROOT}/non_instruction_adapter"
non_instruction_MERGED_DIR  = f"{OUTPUT_ROOT}/non_instruction_merged_model"

instruction_ADAPTER_DIR = f"{OUTPUT_ROOT}/instruction_adapter"
instruction_MERGED_DIR  = f"{OUTPUT_ROOT}/instruction_merged_model"

preference_ADAPTER_DIR = f"{OUTPUT_ROOT}/preference_dpo_adapter"
preference_FINAL_MERGED_DIR   = f"{OUTPUT_ROOT}/preference_dpo_final_merged_model"

for path in [
  OUTPUT_ROOT,
  non_instruction_ADAPTER_DIR,
  non_instruction_MERGED_DIR,
  instruction_ADAPTER_DIR,
  instruction_MERGED_DIR,
  preference_ADAPTER_DIR,
  preference_FINAL_MERGED_DIR,
]:
  os.makedirs(path, exist_ok=True)


In [19]:
# ============================================================
# STAGE 1 DATA: PDF -> raw text dataset
# ============================================================

def build_pdf_dataset(pdf_path: str) -> Dataset:
    pages = extract_pdf_pages(pdf_path)
    records = []

    for page in pages:
        cleaned_text = clean_pdf_text(page["text"])

        for para_id, paragraph in enumerate(cleaned_text.split("\n\n"), start=1):
            paragraph = paragraph.strip()

            if len(paragraph) >= MIN_CHARS_PER_PARAGRAPH:
                records.append({
                    "text": paragraph,
                    "source_page": page["page"],
                    "paragraph_id": para_id,
                })

    if len(records) == 0:
        raise ValueError("No usable paragraph found. Try reducing MIN_CHARS_PER_PARAGRAPH.")

    print("PDF pages extracted:", len(pages))
    print("Paragraph records:", len(records))
    print("\nSample paragraph:\n", records[0]["text"][:700])

    return Dataset.from_list(records)

def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []

    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({
                    "page": page_number,
                    "text": text,
                })

    return pages

def clean_pdf_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by hyphen and newline
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Remove page-number-only lines
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Normalize spaces and paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    paragraphs = []

    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    return "\n\n".join(paragraphs)

In [20]:
# -------------------------
# Helper functions
# -------------------------
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()

def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    This is used at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    - Stage 2 loads STAGE1_MERGED_DIR
    - Stage 3 loads STAGE2_MERGED_DIR
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ], #"embed_tokens", "lm_head"
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth", # was True,
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer

def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"

def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train() #Model Training go here.
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result

def generate_answer(model, tokenizer, instruction: str, input_text: str = "", max_new_tokens: int = 150):
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,    #For SFT Trainig and comment temperature and top_p

            #----------Enable these Parameter for Non-Instruction fine Tuning
            #  do_sample=True,
            # temperature=0.5,
            # top_p=0.9,
            #----------
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens).strip()

def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

In [21]:
non_instruction_dataset = build_pdf_dataset(non_instruction_data_path)

PDF pages extracted: 26
Paragraph records: 25

Sample paragraph:
 Exhibit 10.3 Manufacturing Agreement Between Antares Pharma, Inc. and AMAG Pharmaceuticals, Inc. MANUFACTURING AGREEMENT This Manufacturing Agreement ("Agreement") is made and entered into as of the 20th day of March, 2018 (the "Effective Date") by and between Antares Pharma, Inc., a Delaware corporation, with offices located at 100 Princeton South, Suite 300, Ewing, NJ 08628 ("Antares"), and AMAG Pharmaceuticals, Inc., a Delaware corporation, with a corporate address at 1100 Winter Street, Waltham, MA 02451 ("AMAG"). Antares and AMAG are sometimes referred to herein individually as a "Party" and collectively as the "Parties". Recitals WHEREAS, AMAG is engaged in discovering, developing and 


In [22]:
# -------------------------
# Non Instruction FineTuning
# -------------------------

stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

FastLanguageModel.for_training(stage1_model)

stage1_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    max_steps=STAGE1_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE1_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,
    weight_decay = 0.01,
    max_grad_norm =1.0,  # was 0.3,
    lr_scheduler_type = "cosine",
    seed=SEED,
    warmup_ratio = 0.1,
    # embedding_learning_rate=  STAGE1_EMBR
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=non_instruction_dataset,
    args=stage1_config,
)

train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION  TRAINING")

save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=non_instruction_ADAPTER_DIR,
    merged_dir=non_instruction_MERGED_DIR,
    stage_name="Stage 1",
)
for log in stage1_trainer.state.log_history:
    if 'grad_norm' in log:
        print(f"step {log.get('step')}: grad_norm={log['grad_norm']}")
# #Plot Loss Graph
# losses = [h["loss"] for h in stage1_trainer.state.log_history if "loss" in h]
# losses = np.array(losses)
# k = 5
# roll = np.convolve(losses, np.ones(k)/k, mode="valid")
# roll_x = np.arange(k//2, k//2 + len(roll))   # center the window

# plt.figure(figsize=(9, 4))
# plt.plot(losses, ":", color="#898781", marker=".", label="raw loss")
# plt.plot(roll_x, roll, "-", color="#2a78d6", linewidth=2.5, label="rolling avg (5)")
# plt.axhline(losses.mean(), color="#b4b2a9", linestyle="--", label=f"mean {losses.mean():.3f}")
# plt.xlabel("step"); plt.ylabel("training loss"); plt.legend()
# plt.title(f"range={losses.max()-losses.min():.3f}  drift={roll[-1]-roll[0]:+.3f}")
# plt.show()

del stage1_trainer
del stage1_model
clear_gpu_memory()


==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338
Unsloth: Sample packing skipped (custom data collator detected).


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/25 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25 | Num Epochs = 50 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.021100
2,1.621800
3,1.870400
4,1.859900
5,1.717000
6,2.004900
7,1.736100
8,1.867600
9,1.766600
10,1.669900



STAGE 1 - NON-INSTRUCTION  TRAINING RESULTS
Train time/sec: 247.77
Peak allocated VRAM/GB: 2.294
Peak reserved VRAM/GB: 2.402

Saving Stage 1 adapter...
Stage 1 adapter saved to: /content/unsloth_pharma_merge_reload_outputs/non_instruction_adapter

Merging Stage 1 adapter with base model...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:09<00:00,  9.98s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model`
Stage 1 merged model saved to: /content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model
step 1: grad_norm=9.28233814239502
step 2: grad_norm=3.04951810836792
step 3: grad_norm=7.278594017028809
step 4: grad_norm=4.4605817794799805
step 5: grad_norm=2.945280075073242
step 6: grad_norm=1.1237951517105103
step 7: grad_norm=1.0186810493469238
step 8: grad_norm=9.218594551086426
step 9: grad_norm=4.753239631652832
step 10: grad_norm=2.058753490447998
step 11: grad_norm=1.5366625785827637
step 12: grad_norm=0.8294953107833862
step 13: grad_norm=1.0308016538619995
step 14: grad_norm=0.9846934676170349
step 15: grad_norm=3.2601351737976074
step 16: grad_norm=5.334155082702637
step 17: grad_norm=0.7975012063980103
step 18: grad_norm=0.7577013969421387
step 19: grad_norm=0.73006671667099
step 20: grad_norm=4.69498872756958
step 21: grad_norm=0.9922723174095154
st

In [26]:
# Load BASE model only — no LoRA, no fine-tuning
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=non_instruction_MERGED_DIR,          # "unsloth/tinyllama-bnb-4bit"
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

questions = [
    "What is the Drug that is the subject of the Antares–AMAG Manufacturing Agreement?",
    "Who are the parties to the Manufacturing Agreement?",
    "What is the effective date of the Agreement?",
    "Who are agree parties?",
    "What is the Quality Agreement and when was it originally entered into??",
    "Are there Minimum Orders should be place by AMAG from Antares in each shipment?"
    "What is payment policy?",
    "What is Return and Recall Policy?",
    "State the Effective Date of the Manufacturing Agreement?",
    "What is the \"Transfer Price\" and where is it set out?",
    "Who incorporates the Prefilled Syringe into the Device to produce the finished Product?"
]

# same generation settings as your fixed generate_answer
print("BASE MODEL (before SFT):")

for q in questions:
    print(f"Q: {q}")
    print(f"A: {generate_answer(base_model, base_tokenizer, q, max_new_tokens=250)}")
    print("-" * 80)

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
BASE MODEL (before SFT):
Q: What is the Drug that is the subject of the Antares–AMAG Manufacturing Agreement?
A: The drug in question, **NBUD-BRL** (also known as **Nebulin Autologous Beta‐Lactamase Inhibitor)**. The product was developed by AMAG and manufactured at its facility located within the United States under an agreement with Antares Pharma Inc., a wholly owned subsidiary of Amgen Inc.[1]

[1] https://www.fda.gov/drugs/guidances/ucm350264.htm
</s>
--------------------------------------------------------------------------------
Q: Who 

In [24]:
import glob
from safetensors import safe_open
for p in glob.glob(f"{non_instruction_ADAPTER_DIR}/*.safetensors"):
    with safe_open(p, framework="pt") as f:
        for k in f.keys():
            if "lora_B" in k:
                print(k, "absmax =", f.get_tensor(k).abs().max().item())
                break

base_model.model.model.layers.0.mlp.down_proj.lora_B.weight absmax = 0.002728461753576994


In [25]:
import glob
from safetensors import safe_open
mx, which = 0.0, None
for p in glob.glob(f"{non_instruction_ADAPTER_DIR}/*.safetensors"):
    with safe_open(p, framework="pt") as f:
        for k in f.keys():
            if "lora_B" in k:
                v = f.get_tensor(k).abs().max().item()
                if v > mx: mx, which = v, k
print(f"MAX lora_B: {mx:.6f} ({which})")

MAX lora_B: 0.020409 (base_model.model.model.layers.18.mlp.gate_proj.lora_B.weight)


In [27]:
for name, path in [("BASE", BASE_MODEL_NAME),
                   ("STAGE1_CPT", non_instruction_MERGED_DIR)]:
    m, t = FastLanguageModel.from_pretrained(
        model_name=path, max_seq_length=MAX_SEQ_LENGTH,
        dtype=None, load_in_4bit=True)
    FastLanguageModel.for_inference(m)
    print(f"\n===== [{name}] loaded from: {path} =====")
    print(generate_answer(m, t,
        "What is the Drug that is the subject of the Antares–AMAG Manufacturing Agreement?"))
    del m
    clear_gpu_memory()

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

===== [BASE] loaded from: unsloth/tinyllama-bnb-4bit =====
The drug is the subject of an antitrust agreement between the AMAG Pharmaceuticals and the Antares Pharmacuetical Company. The agreement was signed on 12/30/98, and it has been in effect since then (the date of signing). It is a non-exclusive license to use the patented technology for the manufacture or sale of the product. This means you can make your own version if you want; but not sell any other versions unless they are licensed by us first!

**Question #4** :

What is the name gi

**Instruction Fine Tuning**

In [42]:
# ============================================================
# STAGE 2: Load Stage 1(Non Instruction) merged model -> Instruction Fine Tuning(SFT)
# ============================================================

stage2_model, tokenizer = load_unsloth_model_with_lora(non_instruction_MERGED_DIR)

from datasets import load_dataset

raw = load_dataset("json", data_files=instruction_data_path, split="train")

ALPACA_TEMPLATE = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n{response}"
)
RESPONSE_MARKER = "### Response:\n"

def format_and_mask(example):
    instr = example["instruction"]
    inp = example.get("input", "")
    if inp and inp.strip():
        instr = f"{instr}\n\n### Input:\n{inp}"

    prompt_text = f"### Instruction:\n{instr}\n\n{RESPONSE_MARKER}"
    response_text = example["response"]
    full_text = prompt_text + response_text + tokenizer.eos_token

    # Tokenize prompt and full text with the SAME tokenizer call boundaries
    # by tokenizing prompt alone first, then checking it's a real prefix.
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

    enc = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LENGTH,
                     add_special_tokens=False)
    full_ids = enc["input_ids"]

    # Find the longest matching prefix between prompt_ids and full_ids.
    # In the common case they match exactly. If BPE merges differently
    # across the boundary, walk back until they agree.
    n_prompt = len(prompt_ids)
    while n_prompt > 0 and full_ids[:n_prompt] != prompt_ids[:n_prompt]:
        n_prompt -= 1

    # record whether this example needed a fallback, for auditing
    boundary_clean = (full_ids[:len(prompt_ids)] == prompt_ids)

    labels = list(full_ids)
    for i in range(min(n_prompt, len(labels))):
        labels[i] = -100
    enc["labels"] = labels
    enc["_boundary_clean"] = boundary_clean
    enc["_n_response_tokens"] = len(labels) - n_prompt
    return enc

stage2_dataset = raw.map(format_and_mask, remove_columns=raw.column_names)

# ------------------------------------------------------------
# VALIDATION PASS — run before training, not after
# ------------------------------------------------------------
bad_boundary = []
empty_response = []

for i, ex in enumerate(stage2_dataset):
    trained = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
    n_resp = ex["_n_response_tokens"]
    if not ex["_boundary_clean"]:
        bad_boundary.append((i, raw[i]["instruction"]))
    if n_resp <= 1:  # 1 or 0 unmasked tokens means basically just </s>
        empty_response.append((i, raw[i]["instruction"], tokenizer.decode(trained)))

print(f"Total examples: {len(stage2_dataset)}")
print(f"Examples with boundary mismatch (BPE merge issue): {len(bad_boundary)}")
for i, instr in bad_boundary:
    print(f"  [{i}] {instr}")

print(f"\nExamples with near-empty trained response (<=1 token unmasked): {len(empty_response)}")
for i, instr, decoded in empty_response:
    print(f"  [{i}] {instr!r} -> TRAINED-ON: {decoded!r}")

# sanity check on index 0 as before
ex = stage2_dataset[0]
trained = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
print("\nTRAINED-ON (example 0):", repr(tokenizer.decode(trained)))

# STOP HERE and inspect the printed lists before proceeding to training.
# If empty_response is non-empty, fix those source examples in your
# instruction_data_path JSON (or drop them) before re-running .map().

# Remove the internal debug columns before handing off to SFTTrainer
stage2_dataset = stage2_dataset.remove_columns(["_boundary_clean", "_n_response_tokens"])

# from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(stage2_model)
tokenizer.padding_side = "right"

stage2_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage2_logs",

    max_steps=STAGE2_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE2_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,

    seed=SEED,
)

stage2_trainer = SFTTrainer(
    model=stage2_model,
    processing_class=tokenizer,
    train_dataset=stage2_dataset,
    args=stage2_config,
)

# stage2_trainer = train_on_responses_only(
#     stage2_trainer,
#     instruction_part="### Instruction:\n",
#     response_part="### Response:",     # removed the \n
# )

train_and_measure(stage2_trainer, "STAGE 2 - INSTRUCTION FINE-TUNING")

print("\nStage 2 test answer:")
print(generate_answer(stage2_model, tokenizer, "What is the Drug that is the subject of the Antares–AMAG Manufacturing Agreement.", max_new_tokens=120))


# stage2_model.print_trainable_parameters()
save_adapter_and_merge(
    model=stage2_model,
    tokenizer=tokenizer,
    adapter_dir=instruction_ADAPTER_DIR,
    merged_dir=instruction_MERGED_DIR,
    stage_name="SFT-Instruction Fine Tuning",
)

del stage2_trainer
del stage2_model
clear_gpu_memory()

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338
Total examples: 104
Examples with boundary mismatch (BPE merge issue): 0

Examples with near-empty trained response (<=1 token unmasked): 0

TRAINED-ON (example 0): 'The Drug is 17-alpha hydroxyprogesterone caproate.</s>'


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 104 | Num Epochs = 22 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Step,Training Loss
1,1.838900
2,1.832400
3,1.777500
4,1.831600
5,1.802300
6,1.684700
7,1.594500
8,1.266500
9,1.322700
10,1.364600



STAGE 2 - INSTRUCTION FINE-TUNING RESULTS
Train time/sec: 332.69
Peak allocated VRAM/GB: 3.205
Peak reserved VRAM/GB: 3.322

Stage 2 test answer:
The Drug is 17-alpha hydroxyprogesterone caproate.</s>

Saving SFT-Instruction Fine Tuning adapter...
SFT-Instruction Fine Tuning adapter saved to: /content/unsloth_pharma_merge_reload_outputs/instruction_adapter

Merging SFT-Instruction Fine Tuning adapter with base model...
Detected local model directory: /content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model
Copied tokenizer.model from local model directory
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:09<00:00,  9.01s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/instruction_merged_model`
SFT-Instruction Fine Tuning merged model saved to: /content/unsloth_pharma_merge_reload_outputs/instruction_merged_model


Inference Instruction Tuned Model

In [43]:
# Load Instruction Fine Tuned Model
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=instruction_MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

questions = [
 "What is the Drug that is the subject of the Antares–AMAG Manufacturing Agreement?",
    "Who are the parties to the Manufacturing Agreement?",
    "What is the effective date of the Agreement?",
    "Who are agree parties?",
    "What is the Quality Agreement and when was it originally entered into??",
    "Are there Minimum Orders should be place by AMAG from Antares in each shipment?"
    "What is payment policy?",
    "What is Return and Recall Policy?",
    "State the Effective Date of the Manufacturing Agreement?",
    "What is the \"Transfer Price\" and where is it set out?",
    "Who incorporates the Prefilled Syringe into the Device to produce the finished Product?"
    # "What is the Drug that is the subject of the Antares–AMAG Manufacturing Agreement?",
    # "Define \"Prefilled Syringe.\"",
    # "What is a \"Trainer\" under the agreement?",
    # "Explain what the agreement means by \"Product.\"",
    # "Define the term \"Device\" as used in the agreement.",
    # "Describe the scope of \"Manufacturing Services\" under the agreement."
]

# same generation settings as your fixed generate_answer
print("MODEL After SFT:")

for q in questions:
    print(f"Q: {q}")
    print(f"A: {generate_answer(base_model, base_tokenizer, q, max_new_tokens=250)}")
    print("-" * 80)

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
MODEL After SFT:
Q: What is the Drug that is the subject of the Antares–AMAG Manufacturing Agreement?
A: It is 17-alpha hydroxyprogesterone caproate.</s>
--------------------------------------------------------------------------------
Q: Who are the parties to the Manufacturing Agreement?
A: It is the agreement between AMAG and Antares governing the arrangement between them and the Manufacturer, primarily concerning the manufacture and specification of a Device Product — the 17-alpha hydroxyprogesterone caproate Device — ahead of it being agre

In [38]:
import glob
from safetensors import safe_open
for p in glob.glob(f"{instruction_ADAPTER_DIR}/*.safetensors"):
    with safe_open(p, framework="pt") as f:
        for k in f.keys():
            if "lora_B" in k:
                print(k, "absmax =", f.get_tensor(k).abs().max().item())
                break



base_model.model.model.layers.0.mlp.down_proj.lora_B.weight absmax = 0.0013704110169783235


In [39]:

import glob
from safetensors import safe_open
mx, which = 0.0, None
for p in glob.glob(f"{instruction_ADAPTER_DIR}/*.safetensors"):
    with safe_open(p, framework="pt") as f:
        for k in f.keys():
            if "lora_B" in k:
                v = f.get_tensor(k).abs().max().item()
                if v > mx: mx, which = v, k
print(f"MAX lora_B: {mx:.6f} ({which})")



MAX lora_B: 0.001674 (base_model.model.model.layers.0.self_attn.q_proj.lora_B.weight)


In [40]:
for name, path in [("BASE", BASE_MODEL_NAME),
                   ("STAGE2_SFT", instruction_MERGED_DIR)]:
    m, t = FastLanguageModel.from_pretrained(
        model_name=path, max_seq_length=MAX_SEQ_LENGTH,
        dtype=None, load_in_4bit=True)
    FastLanguageModel.for_inference(m)
    print(f"\n===== [{name}] loaded from: {path} =====")
    print(generate_answer(m, t,
        "What is the Drug that is the subject of the Antares–AMAG Manufacturing Agreement?"))
    del m
    clear_gpu_memory()

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

===== [BASE] loaded from: unsloth/tinyllama-bnb-4bit =====
The drug is the subject of an antitrust agreement between the AMAG Pharmaceuticals and the Antares Pharmacuetical Company. The agreement was signed on 12/30/98, and it has been in effect since then (the date of signing). It is a non-exclusive license to use the patented technology for the manufacture or sale of the product. This means you can make your own version if you want; but not sell any other versions unless they are licensed by us first!

**Question #4** :

What is the name gi